In [0]:
schema = 'fifa_bi_dev.gold_schema'

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = spark.read.table("fifa_bi_dev.silver_schema.silver_world_cup_matches")


result = df.select('round', 'match_stage').dropDuplicates()

result = result.withColumn('rank',when(col('round') == 'Final', 1) 
                           .when(col('round') == 'Play-off for third place', 2)
                           .when(col('round') == 'Match for third place', 2)
                           .when(col('round') == 'Third place', 2)
                           .when(col('round') == 'Semi-finals', 3)
                           .when(col('round') == 'Quarter-finals', 4)
                           .when(col('round').isin('Round of 16', 'First round', 'Preliminary round'), 5)
                           .when(col('match_stage').isin('2nd Group Stage'), 6)
                           .otherwise(8)
                           
                           )
result.display()

In [0]:
result.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{schema}.dim_match_stage')
print(f'table_name: dim_match_stage is updated')